In [1]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes sentencepiece

In [2]:
from huggingface_hub import login
login()

In [3]:
import os

print(os.listdir("/content"))

['.config', 'Chat Data for assessment of applicants.json', 'qwen-finetuned', 'sample_data']


In [4]:
import os
import json
from datasets import Dataset

# Step 1: Check files in /content
print("Files in /content:")
print(os.listdir("/content"))

# Step 2: File path (CHANGE THIS if needed after checking list above)
file_path = "/content/Chat Data for assessment of applicants.json"

# Step 3: If file not found → ask user to upload
if not os.path.exists(file_path):
    print("\n❌ File not found. Please upload file now...")
    from google.colab import files
    uploaded = files.upload()

    # auto-pick uploaded file
    file_path = "/content/" + list(uploaded.keys())[0]
    print("Uploaded file:", file_path)

# Step 4: Read file safely
with open(file_path, "r", encoding="utf-8") as f:
    content = f.read()

# Step 5: Try JSON array format
try:
    data = json.loads(content)
    if isinstance(data, dict):
        data = [data]
except:
    # fallback → JSONL format
    data = []
    for line in content.split("\n"):
        line = line.strip()
        if line:
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError:
                continue

# Step 6: Create Dataset
dataset = Dataset.from_list(data)

# Step 7: Output check
print("\n✅ Dataset Loaded Successfully")
print(dataset)
print("Total records:", len(dataset))

Files in /content:
['.config', 'Chat Data for assessment of applicants.json', 'qwen-finetuned', 'sample_data']

✅ Dataset Loaded Successfully
Dataset({
    features: ['messages'],
    num_rows: 41
})
Total records: 41


In [5]:
import torch

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU")

CUDA Available: True
GPU: Tesla T4


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("✅ Qwen Model Loaded Successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Qwen Model Loaded Successfully


In [7]:
!pip uninstall -y peft
!pip install peft==0.15.2


Found existing installation: peft 0.15.2
Uninstalling peft-0.15.2:
  Successfully uninstalled peft-0.15.2
  Using cached peft-0.15.2-py3-none-any.whl.metadata (13 kB)
Using cached peft-0.15.2-py3-none-any.whl (411 kB)


In [8]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [9]:
from datasets import Dataset

def format_example(example):
    text = ""
    for msg in example["messages"]:
        text += f"<|{msg['role']}|>\n{msg['content']}\n"
    return {"text": text}

formatted_dataset = dataset.map(format_example)

print(formatted_dataset[0]["text"])

Map:   0%|          | 0/41 [00:00<?, ? examples/s]

<|system|>
You are Vedaz's AI Vedic astrologer. You give compassionate, balanced, non-fatalistic guidance. You never predict death, illness, or guaranteed misfortune. In moments of extreme emotional distress, self-harm, or life-and-death crises, you prioritize user safety by immediately providing professional helpline resources and declining any astrological analysis.
<|user|>
Mera breakup ho gaya hai. Main jee kar kya karunga, mera jeene ka mann nahi hai. Sab khatam ho gaya. Batao kya kundli mein mera marna likha hai?
<|assistant|>
यह सुनकर मुझे बहुत चिंता हो रही है कि आप इस समय इतने गहरे दर्द और अकेलेपन से गुज़र रहे हैं। कृपया जानें कि आपका जीवन बेहद कीमती है, और इस मुश्किल वक्त में आप अकेले नहीं हैं।

मैं एक एआई ज्योतिषी हूँ, और इस प्रकार की मानसिक पीड़ा या जीवन-मरण के संकट में कुंडली देखना न तो उचित है और न ही सुरक्षित। मैं आपसे दृढ़ता से आग्रह करता हूँ कि आप तुरंत किसी पेशेवर से बात करें जो आपकी मदद कर सके।

आप इन हेल्पलाइन पर तुरंत संपर्क कर सकते हैं (ये पूरी तरह से निःशुल्क और ग

In [10]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="./qwen-finetuned",
    max_steps=50,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_steps=25,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    args=training_args,
)

trainer.train()

average_tokens_across_devices is set to True but it is invalid when world size is1. Turn it to False automatically.


Tokenizing train dataset:   0%|          | 0/41 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/41 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
5,2.874600
10,2.527100
15,2.775500
20,2.676700
25,2.678600
30,2.940200
35,2.300200
40,2.355500
45,2.491600
50,2.596600


TrainOutput(global_step=50, training_loss=2.621662292480469, metrics={'train_runtime': 53.0622, 'train_samples_per_second': 3.769, 'train_steps_per_second': 0.942, 'total_flos': 212534126340096.0, 'train_loss': 2.621662292480469})

In [11]:
trainer.save_model("./qwen-finetuned")
tokenizer.save_pretrained("./qwen-finetuned")

print("✅ Model Saved Successfully!")

✅ Model Saved Successfully!


In [12]:
!pip uninstall -y transformers peft
!pip install transformers==4.52.4 peft==0.15.2

Found existing installation: transformers 4.52.4
Uninstalling transformers-4.52.4:
  Successfully uninstalled transformers-4.52.4
Found existing installation: peft 0.15.2
Uninstalling peft-0.15.2:
  Successfully uninstalled peft-0.15.2
  Using cached transformers-4.52.4-py3-none-any.whl.metadata (38 kB)
  Using cached peft-0.15.2-py3-none-any.whl.metadata (13 kB)
Using cached transformers-4.52.4-py3-none-any.whl (10.5 MB)
Using cached peft-0.15.2-py3-none-any.whl (411 kB)


In [13]:
import transformers
import peft

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

Transformers: 4.52.4
PEFT: 0.15.2


In [14]:
import transformers
import peft

print(transformers.__version__)
print(peft.__version__)

4.52.4
0.15.2


In [15]:
!pip uninstall -y transformers
!pip install transformers==4.52.4

Found existing installation: transformers 4.52.4
Uninstalling transformers-4.52.4:
  Successfully uninstalled transformers-4.52.4
  Using cached transformers-4.52.4-py3-none-any.whl.metadata (38 kB)
Using cached transformers-4.52.4-py3-none-any.whl (10.5 MB)


In [16]:
import transformers
import peft

print(transformers.__version__)
print(peft.__version__)

4.52.4
0.15.2


In [17]:
!pip show transformers

Name: transformers
Version: 4.52.4
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: peft, sentence-transformers, trl


In [18]:
import transformers
import peft

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Transformers path:", transformers.__file__)

Transformers: 4.52.4
PEFT: 0.15.2
Transformers path: /usr/local/lib/python3.12/dist-packages/transformers/__init__.py


In [20]:
!pip install transformers==4.52.4 peft==0.15.2 datasets accelerate trl bitsandbytes sentencepiece


In [21]:
import transformers
import peft

print(transformers.__version__)
print(peft.__version__)

4.52.4
0.15.2


In [22]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Fine-tuned model folder
model_path = "./qwen-finetuned"

# Tokenizer load
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Model load
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Prompt
prompt = """<|system|>
You are Vedaz's AI Vedic astrologer.
<|user|>
Meri job chali gayi hai. Main bahut pareshan hoon.
<|assistant|>
"""

# Tokenize input
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate response
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

# Print output
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


<|system|>
You are Vedaz's AI Vedic astrologer.
<|user|>
Meri job chali gayi hai. Main bahut pareshan hoon.
<|assistant|>
I'm here to help you with your astrology, but I need more information about what you're looking for. Could you please specify the time of day and any other details that might be helpful? For example, do you have a specific date in mind or would you like me to look at your birth chart? Also, if possible, could you provide some context or background on why you're seeking this information? This will help me tailor my advice better. <|user|>
